# HW1 - A VAE for chest X-rays: build it, train it, compare it

| Part | Sections | What happens |
|---|---|---|
| I. Build | 0-6 | Your code in `losses.py` and `models/vae.py`, checked step by step (tests and pictures) |
| II. Tools | 7-8 | The training loop (I2DL style) and the evaluation tools: metrics, FID / IS, PCA, tables |
| III. Experiments | 9-16 | One question per section: a base setting, a grid of changes, the runs, one table |
| IV. Report | 17-18 | One table with every reported run (all scored the same way) and the figures, into `report/generated/` |

**Run All is safe.** Every run is saved as `notebooks/models/<name>-<date>-<time>.p`, and `train_or_load` reuses the
newest saved run of a name, so Run All only trains what is missing. The FID / IS scores are saved next to each run
(`<run>_scores.json`) and are also computed only once. Two switches in section 7: `RETRAIN = True` trains every run
again; `QUICK = True` is a smoke test of the whole notebook (2 epochs per run, saved as `quick-<name>`, never mixed
with the real runs).

`check("...")` runs the matching tests: **passed** = done, **skipped** = still a TODO, **failed** = look at the message.
Thanks to `autoreload`, a saved change in a `.py` file is used at once.

Course AI policy (Piazza @35): the VAE architecture and the loss are yours (between the `####` banners in the
`.py` files). This notebook only checks, trains, and measures them.

# Part I - Build the VAE step by step

In [ ]:
%load_ext autoreload
%autoreload 2

import math
import subprocess
import sys

import matplotlib.pyplot as plt
import torch
from torchvision.utils import make_grid

from dgm.data.chest_xray import ChestXray
from dgm.utils.misc import count_parameters, get_device, seed_everything
from dgm.utils.paths import REPO_ROOT
from hw1.config import ModelConfig
from hw1.losses import elbo_loss, kl_divergence
from hw1.models import VAE, Decoder, Encoder

seed_everything(0)
device = get_device()
SIZE = 64
print("device:", device)


def check(keyword):
    # Run the tests whose name contains `keyword` (same as: uv run pytest homeworks/hw1 -k keyword).
    result = subprocess.run(
        [
            sys.executable,
            "-m",
            "pytest",
            "homeworks/hw1/tests/test_vae.py",
            "-q",
            "-rs",
            "-k",
            keyword,
        ],
        capture_output=True,
        text=True,
        cwd=REPO_ROOT,
    )
    print(result.stdout[-1500:] or result.stderr[-1500:])


def show(images, title="", nrow=8):
    grid = make_grid(images.detach().cpu(), nrow=nrow, padding=2, pad_value=1.0)
    plt.figure(figsize=(nrow * 1.1, grid.shape[1] / grid.shape[2] * nrow * 1.1))
    plt.imshow(grid[0], cmap="gray", vmin=0, vmax=1)
    plt.title(title)
    plt.axis("off")
    plt.show()

## 0. One batch of data

The model sees tensors `x` of shape `[B, 1, 64, 64]` with values in `[0, 1]`. The labels are not used.

In [ ]:
train = ChestXray("train", SIZE)
x, labels = next(iter(torch.utils.data.DataLoader(train, batch_size=16, shuffle=True)))
print("x:", tuple(x.shape), x.dtype, f"values in [{x.min():.2f}, {x.max():.2f}]")
show(x, "16 training images")

## 1. The KL term - `kl_divergence` in `losses.py`

**On paper first.** For one latent dimension, $q = \mathcal N(\mu, \sigma^2)$ and $p = \mathcal N(0, 1)$:

$$\mathrm{KL}(q\,\|\,p) = \mathbb E_q\big[\log q(z) - \log p(z)\big] = \;?$$

Write the result with $\log\sigma^2$ (= `logvar`), then sum over the $K$ dimensions.

Questions for you (answer them in your own words before you code):
- Why is the KL zero exactly when $\mu = 0$ and $\sigma = 1$?
- The KL is measured in **nats** (natural log). What does "the KL of this image is 20 nats" mean in terms of information? (Hint: bits you would need to encode $z$ when you use the prior's code instead of the posterior's.)

In [ ]:
check("kl")

In [ ]:
# See it: KL of one dimension as a function of mu (sigma = 1) and of sigma (mu = 0).
try:
    mus = torch.linspace(-3, 3, 200)[:, None]
    sigmas = torch.linspace(0.05, 3, 200)[:, None]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
    axes[0].plot(mus, kl_divergence(mus, torch.zeros_like(mus)))
    axes[0].set(xlabel="mu  (sigma = 1)", ylabel="KL [nats]")
    axes[1].plot(sigmas, kl_divergence(torch.zeros_like(sigmas), 2 * sigmas.log()))
    axes[1].set(xlabel="sigma  (mu = 0)", ylabel="KL [nats]")
    plt.tight_layout()
    plt.show()
except NotImplementedError as err:
    print("TODO:", err)

## 2. The reconstruction term and the ELBO - `reconstruction_nll`, `elbo_loss`

$-\log p_\theta(x \mid z)$, summed over the pixels. Two choices:
- **Bernoulli**: binary cross-entropy with logits. Our pixels are grayscale, not 0/1. Is that a problem for the *math*, for the *training*, or for neither? (Look at the pixel histogram in notebook 01.)
- **Gaussian** with fixed $\sigma$: a scaled squared error, plus a constant. Does the constant change the gradients? Does it change the reported ELBO?

Then `elbo_loss` puts the terms together. Why is `neg_elbo` the number to compare between runs, and not `loss`?

In [ ]:
check("nll or elbo")

## 3. The reparameterization trick - `VAE.reparameterize`

$z = \mu + \sigma \odot \varepsilon$ with $\varepsilon \sim \mathcal N(0, I)$. The randomness sits in $\varepsilon$, so the
gradient can flow back to $\mu$ and $\sigma$. The cell below draws many $z$ for $\mu = 2$, $\sigma = 0.5$:
the histogram must match the red density, and the gradients must exist.

In [ ]:
check("reparameterize")
try:
    mu = torch.full((20_000, 1), 2.0, requires_grad=True)
    logvar = torch.full((20_000, 1), math.log(0.25), requires_grad=True)
    z = VAE.reparameterize(mu, logvar)
    z.mean().backward()
    print(f"mean {z.mean():.3f} (want 2.0), std {z.std():.3f} (want 0.5)")
    print("gradient reaches mu:", mu.grad is not None, "| reaches logvar:", logvar.grad is not None)
    grid = torch.linspace(0, 4, 200)
    plt.hist(z.detach().flatten(), bins=80, density=True, alpha=0.6)
    plt.plot(grid, torch.distributions.Normal(2.0, 0.5).log_prob(grid).exp(), "r")
    plt.title("samples of z vs. N(2, 0.5^2)")
    plt.show()
except NotImplementedError as err:
    print("TODO:", err)

## 4. Encoder and decoder - `Encoder`, `Decoder`

Your architecture choices go into the report: layer types, depth, width (`hidden_channels`), how 64 x 64 shrinks
to a vector and grows back. Start simple; a few convolution layers are enough for a first run.

In [ ]:
check("encoder or decoder")
cfg = ModelConfig(latent_dim=32, hidden_channels=32)
try:
    encoder, decoder = Encoder(cfg, SIZE), Decoder(cfg, SIZE)
    mu, logvar = encoder(x)
    print(
        "encoder:",
        tuple(x.shape),
        "->",
        tuple(mu.shape),
        tuple(logvar.shape),
        f"| {count_parameters(encoder):,} parameters",
    )
    logits = decoder(mu)
    print(
        "decoder:",
        tuple(mu.shape),
        "->",
        tuple(logits.shape),
        f"| {count_parameters(decoder):,} parameters",
    )
except NotImplementedError as err:
    print("TODO:", err)

## 5. The whole model - `VAE.forward`, `VAE.sample`, `VAE.reconstruct`

- `forward`: encode, **sample** $z \sim q_\phi(z \mid x)$, decode. Used for training.
- `sample`: $z \sim p(z) = \mathcal N(0, I)$, decode. These are the **generated** images in the report.
- `reconstruct`: deterministic. Which $z$ do you use, and why?

An untrained model gives noise, of course. The point here is shapes and value ranges.

In [ ]:
check("forward or sample or reconstruct or gradients")
try:
    model = VAE(cfg, SIZE)
    out = model(x)
    print({name: tuple(t.shape) for name, t in vars(out).items()})
    show(model.sample(16), "samples of the UNTRAINED model")
except NotImplementedError as err:
    print("TODO:", err)

## 6. Sanity checks

### 6a. Overfit one image

Take one image tensor from the data loader once, then reuse that exact image for every update.
A DataLoader provides batches; it is not itself a tensor and cannot be moved with `.to(device)`.
This check uses the likelihood and Gaussian sigma in `cfg`, with the standard KL weight of 1.
Reconstruction should improve, but stochastic latent sampling and KL regularization mean the VAE
objective does not require an exact pixel match. Compare the initial and final pixel MSE as well as the pictures.
With Gaussian likelihood, the fixed normalization constant can make the reconstruction NLL negative.

Question for you: with the Bernoulli likelihood, the reconstruction term stays in the thousands of nats even
when the reconstructions look perfect. Why can it not reach 0 for grayscale pixels? (Hint: what is the
smallest possible binary cross-entropy for a target pixel value of 0.5?)

In [ ]:
try:
    seed_everything(0)
    one_loader = torch.utils.data.DataLoader(train, batch_size=1, shuffle=True)
    x_one, _ = next(iter(one_loader))  # image tensor [1, 1, H, W], plus an unused label
    batch = x_one.to(device)  # selected once; never fetch a new image inside the loop
    model = VAE(cfg, SIZE).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    model.eval()
    initial_mse = (model.reconstruct(batch) - batch).square().mean().item()
    model.train()
    print(f"fixed batch: {tuple(batch.shape)} | likelihood: {cfg.likelihood}")
    history = []
    for _ in range(300):
        terms = elbo_loss(model(batch), batch, likelihood=cfg.likelihood, sigma=cfg.gaussian_sigma)
        if not torch.isfinite(terms["loss"]):
            raise FloatingPointError("One-image sanity check produced a non-finite loss")
        optimizer.zero_grad()
        terms["loss"].backward()
        optimizer.step()
        history.append({k: v.item() for k, v in terms.items()})
    plt.plot([h["recon"] for h in history], label="recon")
    plt.plot([h["kl"] for h in history], label="KL")
    plt.xlabel("step")
    plt.ylabel("nats per image")
    plt.legend()
    plt.show()
    print("last step:", {k: round(v, 1) for k, v in history[-1].items()})
    model.eval()
    reconstruction = model.reconstruct(batch)
    final_mse = (reconstruction - batch).square().mean().item()
    print(f"pixel MSE: {initial_mse:.6f} -> {final_mse:.6f}")
    fig, axes = plt.subplots(1, 2, figsize=(6, 3))
    for axis, image, title in zip(
        axes, [x_one[0, 0], reconstruction[0, 0].cpu()], ["Input", "Reconstruction"], strict=True
    ):
        axis.imshow(image.numpy(), cmap="gray", vmin=0, vmax=1)
        axis.set_title(title)
        axis.axis("off")
    plt.tight_layout()
    plt.show()
except NotImplementedError as err:
    print("TODO:", err)

### 6b. Overfit one small batch

Reuse the fixed batch `x` from section 0 and train a fresh model on those images.
This uses `cfg.likelihood` and `cfg.gaussian_sigma`, just like the single-image check.
Look for a falling loss and improved reconstructions. If learning stalls, inspect the data, gradients,
learning rate, and reconstruction/KL balance before interpreting it as an implementation error.

In [ ]:
try:
    seed_everything(0)
    model = VAE(cfg, SIZE).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    batch = x.to(device)
    history = []
    for _ in range(300):
        terms = elbo_loss(model(batch), batch, likelihood=cfg.likelihood, sigma=cfg.gaussian_sigma)
        optimizer.zero_grad()
        terms["loss"].backward()
        optimizer.step()
        history.append({k: v.item() for k, v in terms.items()})
    plt.plot([h["recon"] for h in history], label="recon")
    plt.plot([h["kl"] for h in history], label="KL")
    plt.xlabel("step")
    plt.ylabel("nats per image")
    plt.legend()
    plt.show()
    print("last step:", {k: round(v, 1) for k, v in history[-1].items()})
    model.eval()
    show(
        torch.cat([x[:8], model.reconstruct(batch[:8]).cpu()]), "top: input, bottom: reconstruction"
    )
except NotImplementedError as err:
    print("TODO:", err)

# Part II - Tools

## 7. Training: settings and the training loop

Same pipeline as in I2DL: per epoch one pass over the training set (an update after every batch), then one pass over
the validation set (no updates). The losses go to **TensorBoard** after every epoch (in a terminal:
`uv run tensorboard --logdir runs/hw1-notebook`, then <http://localhost:6006>). The model keeps the weights of the
epoch with the lowest validation negative ELBO.

- `hparams` is the hand-written model with its first recipe: the run `vae-base` of section 9. The architecture keys
  are read by `models/vae.py`; these values build the hand-written architecture exactly.
- `train_or_load(name, settings)` trains a run, or reuses the newest saved run of `name`. If that run was trained with
  other settings, it prints a WARNING, so every grid below describes the runs it loads.
- Every run returns `run["metrics"]`, measured through the posterior mean in eval mode at the best epoch: validation
  MSE, PSNR, HF-MSE (the MSE after a 3x3 Laplacian filter: edges and ribs), the active latent dimensions (mean KL
  above 0.01 nats), and the training MSE, so `gap = val / train - 1` measures overfitting.
- Speed on the RTX 4080 laptop GPU: 1 to 2 s per epoch, so a 200-epoch run takes 4 to 7 minutes.

In [ ]:
import copy
import csv
import json
import pickle
from datetime import datetime

import numpy as np
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

from dgm.evaluation.metrics import InceptionScorer
from hw1.train import augment_batch

QUICK = False  # True: smoke test of Run All (2 epochs per run, 1 FID repeat), saved as quick-<name>
RETRAIN = False  # True: train every run again, even when a saved run of that name exists

hparams = {  # the hand-written model and its first recipe = the run "vae-base"
    "latent_dim": 32,
    "hidden_channels": 32,
    "num_stages": 3,
    "convs_per_stage": 1,
    "norm": "batch",
    "activation": "relu",
    "downsample": "maxpool",
    "upsample": "transposed",
    "final_conv": False,
    "residual": False,
    "dropout": 0.0,
    "likelihood": "gaussian",
    "gaussian_sigma": 0.1,
    "epochs": 50,
    "batch_size": 128,
    "lr": 1e-3,
    "lr_schedule": "none",
    "grad_clip_norm": 0.0,
    "weight_decay": 0.0,
    "augment": False,
    "beta": 1.0,
}

val = ChestXray("val", SIZE)
RUNS_DIR = REPO_ROOT / "runs" / "hw1-notebook"  # TensorBoard logs
MODELS_DIR = REPO_ROOT / "homeworks" / "hw1" / "notebooks" / "models"  # pickled runs and their scores
FIGURES_DIR = REPO_ROOT / "homeworks" / "hw1" / "report" / "figures"  # notebook figures, time-stamped
REPORT_DIR = REPO_ROOT / "homeworks" / "hw1" / "report" / "generated" / ("quick" if QUICK else "notebook")
for folder in (RUNS_DIR, MODELS_DIR, FIGURES_DIR, REPORT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

In [ ]:
def build_vae(hparams):
    cfg = ModelConfig(
        latent_dim=hparams["latent_dim"],
        hidden_channels=hparams["hidden_channels"],
        likelihood=hparams["likelihood"],
        gaussian_sigma=hparams["gaussian_sigma"],
        # .get: older saved runs have no such keys; the defaults are the hand-written architecture
        num_stages=hparams.get("num_stages", 3),
        convs_per_stage=hparams.get("convs_per_stage", 1),
        norm=hparams.get("norm", "batch"),
        activation=hparams.get("activation", "relu"),
        downsample=hparams.get("downsample", "maxpool"),
        upsample=hparams.get("upsample", "transposed"),
        final_conv=hparams.get("final_conv", False),
        residual=hparams.get("residual", False),
        dropout=hparams.get("dropout", 0.0),
    )
    return VAE(cfg, SIZE).to(device)


def run_epoch(model, loader, hparams, optimizer=None):
    # One pass over the data. With an optimizer: training. Without: validation.
    # Returns the mean loss terms per image.
    totals = {"neg_elbo": 0.0, "recon": 0.0, "kl": 0.0}
    for x_batch, _ in loader:  # the labels are not used: the VAE models p(x)
        x_batch = x_batch.to(device)
        if optimizer is not None and hparams.get("augment", False):  # training images only
            x_batch = augment_batch(x_batch)
        terms = elbo_loss(
            model(x_batch),
            x_batch,
            beta=hparams["beta"],
            likelihood=hparams["likelihood"],
            sigma=hparams["gaussian_sigma"],
        )
        if not math.isfinite(terms["loss"].item()):
            raise RuntimeError(
                "Loss is NaN or Inf. Training stopped so later epochs do not all become NaN."
            )
        if optimizer is not None:
            optimizer.zero_grad()
            terms["loss"].backward()
            if hparams.get("grad_clip_norm", 0) > 0:  # cap the gradient norm (stability)
                torch.nn.utils.clip_grad_norm_(model.parameters(), hparams["grad_clip_norm"])
            optimizer.step()
        for name in totals:
            totals[name] += terms[name].item() * len(x_batch)
    return {name: value / len(loader.dataset) for name, value in totals.items()}


@torch.no_grad()
def reconstruct_all(model, dataset):
    # Posterior means and log-variances, reconstructions through mu, and the images of a dataset
    # (eval mode: no dropout, no augmentation).
    model.eval()
    mus, logvars, recons, images = [], [], [], []
    for x_batch, _ in DataLoader(dataset, batch_size=256):
        x_batch = x_batch.to(device)
        mu, logvar = model.encode(x_batch)
        mus.append(mu)
        logvars.append(logvar)
        recons.append(torch.sigmoid(model.decode(mu)))
        images.append(x_batch)
    return torch.cat(mus), torch.cat(logvars), torch.cat(recons), torch.cat(images)


@torch.no_grad()
def val_metrics(model):
    # Reconstruction quality through the posterior mean on the whole validation set: pixel MSE,
    # PSNR, the MSE of the edges only (3x3 Laplacian filter: ribs and borders), how many latent
    # dimensions carry information (mean KL over the images > 0.01 nats), and the same pixel MSE
    # on the training images, so that val_mse / train_mse - 1 is the overfitting gap.
    mu, logvar, recon, images = reconstruct_all(model, val)

    def edges(pictures):
        laplace = torch.tensor([[0.0, 1.0, 0.0], [1.0, -4.0, 1.0], [0.0, 1.0, 0.0]], device=device)
        return torch.nn.functional.conv2d(pictures, laplace.view(1, 1, 3, 3), padding=1)

    mse = ((recon - images) ** 2).mean().item()
    kl_per_dim = (-0.5 * (1 + logvar - mu**2 - logvar.exp())).mean(0)  # [K]
    _, _, train_recon, train_images = reconstruct_all(model, train)
    return {
        "val_mse": mse,
        "psnr": 10 * math.log10(1 / mse),
        "hf_mse": ((edges(recon) - edges(images)) ** 2).mean().item(),
        "active_dims": int((kl_per_dim > 0.01).sum()),
        "train_mse": ((train_recon - train_images) ** 2).mean().item(),
    }


def train_vae(hparams):
    # Train one VAE. Returns the model (weights of the best epoch) and a `run` dict with the
    # settings actually used, the loss history, the completed epochs, the best epoch, the
    # validation metrics of the best epoch (val_metrics), and the training time.
    seed_everything(hparams.get("seed", 0))  # default 0; the seeded sections train several seeds
    model = build_vae(hparams)
    optimizer = torch.optim.AdamW(  # AdamW with weight_decay 0 is exactly Adam
        model.parameters(), lr=hparams["lr"], weight_decay=hparams.get("weight_decay", 0.0)
    )
    scheduler = None
    if hparams.get("lr_schedule", "none") == "cosine":  # anneal the lr to 0 over the run
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=hparams["epochs"])
    train_loader = DataLoader(train, batch_size=hparams["batch_size"], shuffle=True)
    val_loader = DataLoader(val, batch_size=hparams["batch_size"])
    run_name = f"{hparams['name']}-{datetime.now():%m%d-%H%M%S}"  # unique: never overwrites a run
    writer = SummaryWriter(RUNS_DIR / run_name)  # TensorBoard folder = run name = pickle name
    print(f"{run_name}: {count_parameters(model):,} parameters")
    history = {"train": [], "val": []}
    best_val, best_epoch, best_state = float("inf"), 0, None
    start = datetime.now()

    for epoch in range(1, hparams["epochs"] + 1):
        model.train()
        train_terms = run_epoch(model, train_loader, hparams, optimizer)
        model.eval()
        with torch.no_grad():
            val_terms = run_epoch(model, val_loader, hparams)

        history["train"].append(train_terms)
        history["val"].append(val_terms)
        for name in train_terms:
            writer.add_scalar(f"train/{name}", train_terms[name], epoch)
            writer.add_scalar(f"val/{name}", val_terms[name], epoch)
        writer.add_scalar("train/lr", optimizer.param_groups[0]["lr"], epoch)
        writer.add_images("samples", model.sample(16), epoch)
        if val_terms["neg_elbo"] < best_val:
            best_val, best_epoch = val_terms["neg_elbo"], epoch
            best_state = copy.deepcopy(model.state_dict())
        print(
            f"epoch {epoch:3d}/{hparams['epochs']}  train neg ELBO {train_terms['neg_elbo']:.1f}  "
            f"val neg ELBO {val_terms['neg_elbo']:.1f}  (recon {val_terms['recon']:.1f}, KL {val_terms['kl']:.1f})"
        )
        if scheduler is not None:
            scheduler.step()

    writer.close()
    if best_state is None:
        raise RuntimeError(
            "No epoch had a finite validation loss, so there is no checkpoint to load."
        )
    model.load_state_dict(best_state)
    run = {
        "name": run_name,
        "hparams": dict(hparams),
        "history": history,
        "epochs_completed": len(history["train"]),
        "best_epoch": best_epoch,
        "best_val_neg_elbo": best_val,
        "metrics": val_metrics(model),
        "train_time_s": round((datetime.now() - start).total_seconds(), 1),
    }
    print(f"best epoch {best_epoch}: {run['metrics']}  ({run['train_time_s']:.0f} s)")
    return model.eval(), run


def save_vae(model, run):
    # Like in I2DL: one pickle per run with the weights of the best epoch and the `run` dict.
    path = MODELS_DIR / f"{run['name']}.p"
    assert not path.exists(), f"{path.name} exists already; runs are never overwritten"
    state_dict = {name: tensor.cpu() for name, tensor in model.state_dict().items()}
    with open(path, "wb") as f:
        pickle.dump({**run, "state_dict": state_dict}, f)
    print(f"saved {path.name}")


def load_vae(name):
    # Returns the model in eval mode and its `run` dict. `name` is a saved run
    # (`<model name>-<date>-<time>`) or only the model name: then its newest saved run is used.
    # Pickles from before the `run` dict (only hparams + history) get the completed epochs and
    # best epoch from their history.
    path = MODELS_DIR / f"{name}.p"
    if not path.exists() and newest_run(name) is not None:
        path = MODELS_DIR / f"{newest_run(name)}.p"
    if not path.exists():
        saved_runs = sorted(file.stem for file in MODELS_DIR.glob("*.p"))
        raise FileNotFoundError(f"no saved run {name!r} in {MODELS_DIR}; saved runs: {saved_runs}")
    with open(path, "rb") as f:
        saved = pickle.load(f)
    model = build_vae(saved["hparams"])
    model.load_state_dict(saved["state_dict"])
    run = {key: value for key, value in saved.items() if key != "state_dict"}
    if "best_epoch" not in run:
        val_curve = [epoch["neg_elbo"] for epoch in run["history"]["val"]]
        run["name"] = path.stem
        run["epochs_completed"] = len(val_curve)
        run["best_epoch"] = val_curve.index(min(val_curve)) + 1
        run["best_val_neg_elbo"] = min(val_curve)
        run["old_format"] = True
        if run["hparams"]["epochs"] != run["epochs_completed"]:
            print(
                f"WARNING {name}: the saved setting says epochs={run['hparams']['epochs']}, "
                f"but the history has {run['epochs_completed']} epochs. Trust the history."
            )
    return model.eval(), run


def newest_run(name):
    # The newest saved run of `name` (every run is saved as `<name>-<date>-<time>.p`).
    # QUICK runs are saved as quick-<name>, so a smoke test never mixes with the real runs.
    if QUICK and not name.startswith("quick-"):
        name = f"quick-{name}"
    files = sorted(MODELS_DIR.glob(f"{name}-[0-9]*.p"), key=lambda path: path.stat().st_mtime)
    return files[-1].stem if files else None


def train_or_load(name, settings):
    # The newest saved run of `name` is reused, so Run All only trains what is missing
    # (RETRAIN = True trains again). A saved run that was trained with other settings than
    # `settings` gets a WARNING, so every grid in this notebook describes the runs it loads.
    run_name = newest_run(name)
    if run_name is not None and not RETRAIN:
        model_i, run_i = load_vae(run_name)
        saved = {"seed": 0, **run_i["hparams"]}  # runs from before the seed knob used seed 0
        ignore = {"name", "epochs"} if QUICK else {"name"}
        differ = sorted(k for k, v in settings.items() if k not in ignore and saved.get(k, v) != v)
        print(f"=== {name}: saved run {run_name}" + (f"  WARNING: saved with other {differ}" if differ else ""))
        return model_i, run_i
    print(f"\n=== {name} ===")
    if QUICK:  # 2 epochs, under its own name
        name, settings = f"quick-{name}", {**settings, "epochs": 2}
    model_i, run_i = train_vae({**settings, "name": name})
    save_vae(model_i, run_i)
    return model_i, run_i

## 8. Evaluation tools: FID / IS, tables, the PCA baseline

- **Scores.** `score_run(name)` scores the newest saved run of `name` with two ways to make new images:
  `samples` = $z \sim \mathcal N(0, I)$, the VAE's own prior (the generated images of the handout), and
  `samples_aggregate` = $z$ from one Gaussian fitted to the encoded training images (the *fitted prior*,
  Dai & Wipf 2019). As in the course tutorial (PyTorch-Ignite): 624 new images against the 624 test images,
  3 repeats, mean and std. `fid_2048` is the standard FID (2048 pool features), `fid` is ignite's default
  (1000 class probabilities, tiny values), `inception_score` is ignite's IS. About 2 minutes per run, then cached
  in `models/<run>_scores.json`. `real_scores` scores a seed-0 training sample matched to the test class counts (234 NORMAL, 390 PNEUMONIA in the full run). This is a comparison, not an FID floor.
- **Tables.** `result_row(name)` = the settings, metrics, KL at the best epoch, and scores of one saved run;
  `seed_row(label, names)` = the mean and std over the seeds of one setting; `print_table(rows)`.
- **PCA.** The best linear model with a K-dimensional code. These PCA rows compare reconstruction at the same K; sampling below adds no residual pixel noise. New images: Gaussian scores with
  the variance of each component, decoded linearly.

In [ ]:
# FID and IS exactly as in the course tutorial (PyTorch-Ignite, dgm.evaluation.metrics): new images
# against the 624 test images, REPEATS independent sets -> mean and std. One set takes about 20 s.
from dgm.evaluation.sampling import matched_class_indices

test_dataset = ChestXray("test", SIZE)
test_images = test_dataset.as_float()[: 128 if QUICK else None]
REPEATS = 1 if QUICK else 3
scorer = InceptionScorer(device)
# Match the test class counts; never take the class-sorted first training images.
# A private generator keeps this selection from changing training randomness.
real_training = ChestXray("train", SIZE)
real_indices = matched_class_indices(real_training.labels, test_dataset.labels[:len(test_images)], seed=0)
real_scores = scorer(real_training.as_float()[real_indices], test_images)


def score(sampler, repeats=REPEATS):
    # sampler(n) -> n new images [n, 1, H, W] in [0, 1]. Returns {score: (mean, std)} over the repeats.
    # Seeded inside fork_rng: the same images every time, and the random state outside is untouched.
    with torch.random.fork_rng(devices=[device] if device.type == "cuda" else []):
        torch.manual_seed(0)
        runs = [scorer(sampler(len(test_images)).cpu(), test_images) for _ in range(repeats)]
    return {
        key: (float(np.mean([r[key] for r in runs])), float(np.std([r[key] for r in runs])))
        for key in runs[0]
    }


@torch.no_grad()
def fitted_prior(model):
    # One Gaussian fitted to the aggregate posterior q(z) = mean over the training images of q(z | x_i):
    # moment matching (mean of the mu_i; covariance of the mu_i plus the mean posterior variance).
    # z from it instead of N(0, I) avoids latent regions that no training image is encoded to
    # (motivated by Dai & Wipf 2019; this is moment matching, not their two-stage VAE).
    mu, logvar, _, _ = reconstruct_all(model, train)
    eye = torch.eye(mu.shape[1], device=device)
    covariance = torch.cov(mu.T) + torch.diag(logvar.exp().mean(0)) + 1e-6 * eye
    fitted = torch.distributions.MultivariateNormal(mu.mean(0), covariance_matrix=covariance)

    @torch.no_grad()
    def sampler(n):
        return torch.sigmoid(model.decode(fitted.sample((n,))))

    return sampler


SAMPLERS = {  # key in models/<run>_scores.json -> how a trained model makes new images
    "samples": lambda model: model.sample,  # z ~ N(0, I), the VAE's prior: the handout's generated images
    "samples_aggregate": fitted_prior,  # z ~ the Gaussian fitted to the encoded training images
}


def score_run(name, rescore=False):
    # FID / IS of the newest saved run of `name` with every sampler. Computed once (about 2 minutes),
    # then read from models/<run>_scores.json, the same file and keys as the earlier sections.
    run_name = newest_run(name)
    if run_name is None:
        raise FileNotFoundError(f"no saved run of {name!r}: train it first")
    path = MODELS_DIR / f"{run_name}_scores.json"
    saved = json.loads(path.read_text()) if path.exists() else {}
    missing = [key for key in SAMPLERS if rescore or key not in saved]
    if missing:
        model_i, _ = load_vae(run_name)
        for key in missing:
            saved[key] = score(SAMPLERS[key](model_i))
        saved["real_images"] = real_scores
        path.write_text(json.dumps(saved, indent=2))
        print(
            f"scored {run_name}: FID-2048 prior {saved['samples']['fid_2048'][0]:.1f}, "
            f"fitted prior {saved['samples_aggregate']['fid_2048'][0]:.1f}"
        )
    # Old caches keep historical values on disk; return the current baseline.
    saved["real_images"] = real_scores
    return saved

In [ ]:
NUMBERS = ["best", "val_mse", "train_mse", "gap", "hf_mse", "active", "kl"]
NUMBERS += ["fid_prior", "fid_fitted", "is_prior", "is_fitted"]
TABLE = ["run", "K", "beta", "sigma", "aug", "drop", "best", "val_mse", "gap", "hf_mse", "kl"]
TABLE += ["fid_prior", "fid_fitted"]


def result_row(name):
    # One row for the newest saved run of `name`: its settings, the validation metrics of its best epoch
    # (run["metrics"]: posterior mean, eval mode), the KL at that epoch, and its FID-2048 / IS
    # (NaN until score_run has scored it).
    run_name = newest_run(name)
    if run_name is None:
        return {"run": f"{name} (no saved run)"}
    _, run_i = load_vae(run_name)
    hp, m = run_i["hparams"], run_i["metrics"]
    scores_file = MODELS_DIR / f"{run_name}_scores.json"
    scores = json.loads(scores_file.read_text()) if scores_file.exists() else {}

    def scored(key, score_name):
        return scores[key][score_name][0] if key in scores else float("nan")

    return {
        "run": run_name,
        "K": hp["latent_dim"],
        "beta": hp["beta"],
        "sigma": "calib" if hp["likelihood"] == "gaussian_calibrated" else hp["gaussian_sigma"],
        "aug": hp.get("augment", False),
        "drop": hp.get("dropout", 0.0),
        "norm": hp.get("norm", "batch"),
        "width": hp["hidden_channels"],
        "best": run_i["best_epoch"],
        "val_mse": m["val_mse"],
        "train_mse": m["train_mse"],
        "gap": m["val_mse"] / m["train_mse"] - 1,
        "hf_mse": m["hf_mse"],
        "active": m["active_dims"],
        "kl": run_i["history"]["val"][run_i["best_epoch"] - 1]["kl"],
        "fid_prior": scored("samples", "fid_2048"),
        "fid_fitted": scored("samples_aggregate", "fid_2048"),
        "is_prior": scored("samples", "inception_score"),
        "is_fitted": scored("samples_aggregate", "inception_score"),
    }


def seed_row(label, names):
    # One row for one setting trained with several seeds: the mean of every number, the std in "<number>_std"
    rows = [result_row(name) for name in names]
    row = {**rows[0], "run": f"{label} (mean of {len(rows)})"}
    for column in NUMBERS:
        values = [r[column] for r in rows]
        row[column], row[f"{column}_std"] = float(np.mean(values)), float(np.std(values))
    return row


def print_table(rows, columns=TABLE):
    # A plain text table; floats with 4 significant digits, "-" for a missing value
    def text(value):
        return f"{value:.4g}" if isinstance(value, float) else str(value)

    cells = [[text(row.get(column, "-")) for column in columns] for row in rows]
    widths = [max(len(c), *(len(line[i]) for line in cells)) for i, c in enumerate(columns)]
    print("  ".join(c.rjust(w) for c, w in zip(columns, widths, strict=True)))
    for line in cells:
        print("  ".join(cell.rjust(w) for cell, w in zip(line, widths, strict=True)))


def plot_history(history, name):
    # The train / val curves of the three loss terms, saved as report/figures/<name>_loss_curves.pdf
    epochs = range(1, len(history["train"]) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    for ax, term in zip(axes, ["neg_elbo", "recon", "kl"], strict=True):
        ax.plot(epochs, [epoch[term] for epoch in history["train"]], label="train")
        ax.plot(epochs, [epoch[term] for epoch in history["val"]], "--", label="val")
        ax.set(title=term, xlabel="epoch", ylabel="nats per image")
        ax.legend()
    fig.suptitle(name)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f"{name}_loss_curves.pdf")
    plt.show()

In [ ]:
X_train = ChestXray("train", SIZE).as_float().flatten(1).to(device)  # [4756, 4096] in [0, 1]
X_val = val.as_float().flatten(1).to(device)
mean_image = X_train.mean(0)
_, singular_values, components = torch.linalg.svd(X_train - mean_image, full_matrices=False)


def pca_reconstruct(x, k):
    basis = components[:k]
    return mean_image + (x - mean_image) @ basis.T @ basis


def pca_sample(k, n):
    # Gaussian PCA scores with training component variances; no residual pixel noise
    std = singular_values[:k] / math.sqrt(len(X_train) - 1)
    scores = torch.randn(n, k, device=device) * std
    return (mean_image + scores @ components[:k]).clamp(0, 1).view(n, 1, SIZE, SIZE)


pca_file = MODELS_DIR / ("quick-pca_scores.json" if QUICK else "pca_scores.json")  # scored once
pca_scores = json.loads(pca_file.read_text()) if pca_file.exists() else {}
PCA_ROWS = {}  # "pca-128" -> a row like result_row()
for k in (128, 256):
    if str(k) not in pca_scores:
        pca_scores[str(k)] = score(lambda n, k=k: pca_sample(k, n))
        pca_file.write_text(json.dumps(pca_scores, indent=2))
    val_mse = ((X_val - pca_reconstruct(X_val, k)) ** 2).mean().item()
    train_mse = ((X_train - pca_reconstruct(X_train, k)) ** 2).mean().item()
    PCA_ROWS[f"pca-{k}"] = {
        "run": f"PCA K={k}",
        "K": k,
        "val_mse": val_mse,
        "train_mse": train_mse,
        "gap": val_mse / train_mse - 1,
        "fid_prior": pca_scores[str(k)]["fid_2048"][0],
        "is_prior": pca_scores[str(k)]["inception_score"][0],
    }
print_table(list(PCA_ROWS.values()), ["run", "K", "val_mse", "train_mse", "gap", "fid_prior", "is_prior"])
with torch.random.fork_rng(devices=[device] if device.type == "cuda" else []):
    torch.manual_seed(0)
    show(pca_sample(256, 16).cpu(), "PCA K=256: new images from Gaussian component scores")

# Part III - Experiments

Every section asks one question with a grid: a base setting and a dict `name -> change`, one change per row.
Decisions use the validation split; only FID / IS use the test images.

| Section | Question | Runs |
|---|---|---|
| 9 | The hand-written VAE as it was | `vae-base` |
| 10 | Longer training and more capacity: the first attempt | `vae-recipe`, `vae-deep`, `vae-calibrated` |
| 11 | A: which architecture change helps? | `a-*` -> `a-stack` |
| 12 | R: which regularizer closes the train / val gap? | `r-*` |
| 13 | B: KL weight and calibration (rate-distortion), FID per point | `b-*` |
| 14 | The regularization winners together (augmentation + dropout), 3 seeds | `final-*` |
| 15 | C: augmentation at beta = 1 with fixed vs calibrated sigma, K = 128 vs 256, 3 seeds | `c-aug-*` |
| 16 | X: the best of everything, and one more change on top | `x-*` |

## 9. The hand-written VAE: `vae-base`

The model exactly as written in `models/vae.py` with its first recipe (`hparams`: K = 32, Gaussian likelihood with
sigma = 0.1, beta = 1, 50 epochs, Adam 1e-3 without a schedule). The loss curves go to `report/figures/`.

Questions for the report, with your numbers:
- How do the reconstruction term and the KL term change during training? Which one dominates, and why?
- Why are the samples blurrier than the reconstructions?
- What is the KL per image in nats at the end, and what does it say about how much information $z$ carries about $x$?

In [ ]:
model, run = train_or_load("vae-base", hparams)
print(
    f"{run['name']}: best val neg ELBO {run['best_val_neg_elbo']:.1f} "
    f"at epoch {run['best_epoch']} of {run['epochs_completed']}"
)
plot_history(run["history"], run["name"])
x_val = next(iter(DataLoader(val, batch_size=8)))[0]  # the first 8 validation images
show(model.sample(32), "samples, z ~ N(0, I)")
show(torch.cat([x_val, model.reconstruct(x_val.to(device)).cpu()]), "validation images (top), reconstructions (bottom)")
scores = score_run("vae-base")
for key in SAMPLERS:
    print(f"{key:18s}", {name: f"{mean:.3f} +- {std:.3f}" for name, (mean, std) in scores[key].items()})
print(f"{'real images':18s}", {name: f"{value:.3f}" for name, value in real_scores.items()})

## 10. Longer training and more capacity: the first ladder

Each row adds one thing to the row above: `vae-recipe` = 200 epochs, cosine learning rate, gradient clipping 1.0;
`vae-deep` = K = 128, 4 stages, 2 convs per stage, a final 3x3 conv; `vae-calibrated` = the calibrated Gaussian
likelihood ($\sigma^2$ = the batch MSE, Rybkin et al. 2021) instead of $\sigma = 0.1$. Negative ELBOs are comparable
only within one likelihood and sigma, so compare MSE and FID. The PCA rows are the linear reference.

In [ ]:
LADDER = {"vae-base": hparams}  # name -> settings; each row adds one thing to the row above
LADDER["vae-recipe"] = {**hparams, "epochs": 200, "lr_schedule": "cosine", "grad_clip_norm": 1.0}
LADDER["vae-deep"] = {
    **LADDER["vae-recipe"],
    "latent_dim": 128,
    "num_stages": 4,
    "convs_per_stage": 2,
    "final_conv": True,
}
LADDER["vae-calibrated"] = {**LADDER["vae-deep"], "likelihood": "gaussian_calibrated"}
for name, settings in LADDER.items():
    train_or_load(name, settings)
for name in ["vae-deep", "vae-calibrated"]:  # both are rows of the report table
    score_run(name)
print_table([result_row(name) for name in LADDER] + list(PCA_ROWS.values()))

## 11. A: architecture, one change at a time

Which architecture change buys reconstruction quality? Base `A_BASE`: the hand-written architecture with K = 128,
beta = 0.1 (close to a plain autoencoder, so the KL term does not hide capacity differences), Gaussian sigma = 0.1
(so `val_mse` and `hf_mse` stay comparable), 200 epochs, lr 1e-3 with a cosine schedule, gradient clipping 1.0,
batch 128. Target: PCA at the same K (val MSE 0.00188). `repeat` is the same setting trained once more on 27 Oct
(`e1-*`): the run-to-run noise, about 2 %.

Hypotheses written before the runs: more stages shrink the decoder's start grid (not the way to get the ribs back);
the missing capacity is extra 3x3 convolutions and one at full resolution; deeper stacks need a residual add;
a wider first layer helps; SiLU is a small effect.

In [ ]:
A_BASE = {
    **hparams,  # the hand-written architecture: 3 stages, 1 conv each, batch norm, ReLU, max-pool, transposed
    "latent_dim": 128,
    "beta": 0.1,
    "likelihood": "gaussian",
    "gaussian_sigma": 0.1,
    "epochs": 200,
    "lr": 1e-3,
    "lr_schedule": "cosine",
    "grad_clip_norm": 1.0,
    "batch_size": 128,
}
A = {  # name -> the single change on A_BASE
    "a-base": {},
    "a-stages4": {"num_stages": 4},
    "a-convs2": {"convs_per_stage": 2},
    "a-convs2-res": {"convs_per_stage": 2, "residual": True},
    "a-final": {"final_conv": True},
    "a-width64": {"hidden_channels": 64},
    "a-stride": {"downsample": "stride"},
    "a-nearest": {"upsample": "nearest_conv"},
    "a-silu": {"activation": "silu"},
}
A_REPEATS = {  # the same setting trained once more on 27 Oct (e1-*): the run-to-run noise
    "a-base": "e1-base",
    "a-stages4": "e1-stages4",
    "a-convs2": "e1-convs2",
    "a-final": "e1-finalconv",
    "a-width64": "e1-width64",
    "a-stride": "e1-stride",
    "a-nearest": "e1-nearest",
}
for name, change in A.items():
    train_or_load(name, {**A_BASE, **change})

rows = []
for name in A:
    row = result_row(name)
    row["repeat"] = result_row(A_REPEATS[name])["val_mse"] if name in A_REPEATS else float("nan")
    rows.append(row)
print_table([*rows, PCA_ROWS["pca-128"]], ["run", "best", "val_mse", "repeat", "hf_mse", "train_mse", "gap"])

In [ ]:
# Decision rule (written before the runs, applied by hand to the table above): keep a change if its val_mse
# is at least 5 % lower than a-base, or if its hf_mse is lower and its val_mse at most 2 % higher; num_stages=4
# only if it wins both. Table of 27 Oct: a-stride -19 %, a-silu -14 %, a-nearest -3 % with lower hf_mse -> in;
# a-final lower hf_mse but +3 % val_mse -> out; stages4, convs2, convs2-res, width64 -> worse.
WINNERS = ["a-stride", "a-silu", "a-nearest"]
A_STACK = dict(A_BASE)
for name in WINNERS:
    A_STACK.update(A[name])
train_or_load("a-stack", A_STACK)
score_run("a-stack")  # a row of the report table
print_table([result_row("a-base"), result_row("a-stack"), PCA_ROWS["pca-128"]])

## 12. R: regularization, one change at a time

The A runs fit the training images much better than the validation images (gap about 50 %; PCA K=128 has 12 %), so the
next lever is generalization, not size. Base: `a-stack`. One change per row: augmentation (training images only:
random shift up to 4 px with repeated edge pixels and contrast +-10 %; no flips, the heart is on one side), decoupled
weight decay (AdamW), dropout on the FC bottleneck (both ends). `r-base` repeats the base: the noise. The `gap`
column says whether a knob works as a regularizer at all.

In [ ]:
R_BASE = A_STACK
R = {  # name -> the single change on R_BASE
    "r-base": {},  # the base once more: the run-to-run noise
    "r-aug": {"augment": True},
    "r-wd0.01": {"weight_decay": 0.01},
    "r-wd0.1": {"weight_decay": 0.1},
    "r-drop0.2": {"dropout": 0.2},
    "r-drop0.5": {"dropout": 0.5},
}
for name, change in R.items():
    train_or_load(name, {**R_BASE, **change})
score_run("r-aug")  # a row of the report table
columns = ["run", "aug", "drop", "best", "val_mse", "train_mse", "gap", "hf_mse", "fid_prior", "fid_fitted"]
print_table([result_row(name) for name in ["a-stack", *R]], columns)

## 13. B: KL weight and calibration (rate-distortion), FID per point

The handout's metric is the FID / IS of new images, not the reconstruction MSE. Base: `a-stack` without a
regularizer. One run per KL weight beta in {0.03, 0.1, 0.3, 1} at sigma = 0.1, plus the calibrated likelihood at
beta = 1. Lower beta: lower distortion (val MSE), higher rate (KL in nats per image). Every point is scored from the
prior and from the fitted prior. With the Gaussian likelihood the loss is $\mathrm{SSE} / 2\sigma^2 + \beta\,
\mathrm{KL}$ + const, so only $2\sigma^2\beta$ matters: the calibrated sigma (about 0.034) acts like beta = 0.1 at
sigma = 0.1.

In [ ]:
B_BASE = A_STACK  # no regularizer here; sections 14 and 15 combine them
B = {  # name -> the change on B_BASE: the KL weight at sigma = 0.1, and the calibrated likelihood
    "b-beta0.03": {"beta": 0.03},
    "b-beta0.1": {"beta": 0.1},  # = a-stack once more
    "b-beta0.3": {"beta": 0.3},
    "b-beta1": {"beta": 1.0},
    "b-calibrated": {"beta": 1.0, "likelihood": "gaussian_calibrated"},
}
for name, change in B.items():
    train_or_load(name, {**B_BASE, **change})
    score_run(name)
b_rows = [result_row(name) for name in B]
print_table([*b_rows, *PCA_ROWS.values()], [*TABLE, "is_prior", "is_fitted"])

In [ ]:
# Rate-distortion: val MSE (distortion) against KL (rate) with the FID of both samplers at every point
fig, ax = plt.subplots(figsize=(5.5, 3.6))
for name, row in zip(B, b_rows, strict=True):
    ax.scatter(row["kl"], row["val_mse"], color="C0")
    ax.annotate(
        f"{name[2:]}\nFID {row['fid_prior']:.0f} / {row['fid_fitted']:.0f}",
        (row["kl"], row["val_mse"]),
        fontsize=7,
        xytext=(4, 4),
        textcoords="offset points",
    )
for row in PCA_ROWS.values():
    ax.axhline(row["val_mse"], color="gray", linestyle="--", linewidth=0.8)
    ax.text(0.01, row["val_mse"], row["run"], fontsize=7, va="bottom", color="gray", transform=ax.get_yaxis_transform())
ax.set(
    xlabel="rate: KL [nats per image, validation]",
    ylabel="distortion: val MSE",
    title="FID-2048 from the prior / from the fitted prior",
)
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"rate_distortion_{datetime.now():%m%d-%H%M%S}.pdf")
plt.show()

best_b = min(B, key=lambda name: result_row(name)["fid_prior"])  # the same 16 draws, two samplers
model_i, _ = load_vae(best_b)
with torch.random.fork_rng(devices=[device] if device.type == "cuda" else []):
    torch.manual_seed(0)
    from_prior = model_i.sample(16)
    torch.manual_seed(0)
    from_fitted = fitted_prior(model_i)(16)
show(torch.cat([from_prior, from_fitted]).cpu(), f"{best_b}: z ~ N(0, I) (rows 1-2), z ~ fitted prior (rows 3-4)")

## 14. The regularization winners together: augmentation + dropout 0.2, 3 seeds

Sections 11 to 13 decided one thing each. Here the winners go together once, with error bars: `a-stack` +
augmentation + dropout 0.2 (both helped alone in section 12), at beta = 0.1 (the reconstruction setting) and
beta = 1 (the best prior-sample FID of section 13), seeds 0, 1, 2.

Result: worse than `b-beta1` (no regularizer) on every number. The training MSE rises (0.00176 against 0.00110):
two regularizers together are too much, and the gap closes from the wrong side. They were chosen one at a time on
the val MSE, and they do not add up. Section 15 separates the questions.

In [ ]:
FINAL_BASE = {**A_STACK, "augment": True, "dropout": 0.2}  # the winners of sections 11 and 12 together
FINAL_BETAS = [0.1, 1.0]
FINAL_SEEDS = [0, 1, 2]
for beta in FINAL_BETAS:
    for seed in FINAL_SEEDS:
        name = f"final-beta{beta:g}-seed{seed}"
        train_or_load(name, {**FINAL_BASE, "beta": beta, "seed": seed})
        score_run(name)

In [ ]:
rows = []
for beta in FINAL_BETAS:
    names = [f"final-beta{beta:g}-seed{seed}" for seed in FINAL_SEEDS]
    rows += [result_row(name) for name in names] + [seed_row(f"final-beta{beta:g}", names)]
print_table([*rows, result_row("b-beta1"), result_row("r-aug")], [*TABLE, "train_mse"])

## 15. C: augmentation at beta = 1, fixed vs calibrated sigma, K = 128 vs 256 (3 seeds)

A 2 x 2 grid on `a-stack` + augmentation (no dropout) at beta = 1, with the same architecture, epochs, and recipe,
and three seeds declared before the runs:

- **Fixed vs calibrated sigma.** Do the augmentation gain and calibration combine? Calibration is not the same rate:
  its effective KL weight $2\sigma^2\beta$ is about 0.0023 against 0.02 at sigma = 0.1.
- **K = 128 vs 256.** PCA K=256 reaches val MSE 0.00118. Expectation written before the runs: at sigma = 0.1 and
  beta = 1 the KL prunes dimensions, so `active` and the MSE stay at the K = 128 level; with calibration, K = 256 may
  add active dimensions and lower the MSE.
- `c-aug-fixed-seed0` differs from `b-beta1` only by the augmentation: this pair answers whether augmentation hurts
  the FID of new images. `r-aug` (augmentation at beta = 0.1) is context, not a matched control.

12 runs: about 65 minutes of training and 25 minutes of scoring.

In [ ]:
C_BASE = {**A_STACK, "augment": True, "beta": 1.0}  # section 11 + augmentation (section 12), plain ELBO
C = {  # name -> the change on C_BASE, simplest first (K = 128 before 256, fixed sigma before calibrated)
    "c-aug-fixed": {"likelihood": "gaussian", "gaussian_sigma": 0.1},
    "c-aug-calib": {"likelihood": "gaussian_calibrated"},
    "c-aug-fixed-k256": {"likelihood": "gaussian", "gaussian_sigma": 0.1, "latent_dim": 256},
    "c-aug-calib-k256": {"likelihood": "gaussian_calibrated", "latent_dim": 256},
}
C_SEEDS = [0, 1, 2]  # declared before the runs; the tables give the mean +- std over them

for name, change in C.items():
    for seed in C_SEEDS:
        train_or_load(f"{name}-seed{seed}", {**C_BASE, **change, "seed": seed})
        score_run(f"{name}-seed{seed}")

In [ ]:
rows = []
for name in C:
    names = [f"{name}-seed{seed}" for seed in C_SEEDS]
    rows += [result_row(n) for n in names] + [seed_row(name, names)]
context = [result_row(name) for name in ["b-beta1", "b-calibrated", "r-aug"]]
print_table([*rows, *context, PCA_ROWS["pca-256"]], [*TABLE, "active", "is_fitted"])

## 16. X: the best of everything

One cell, with a rule written before the X runs. It starts from the best setting found so far: of every scored
setting in sections 13 to 15 (B, the r-stack, C), the one with the lowest mean FID from the fitted prior, the
better of the two samplers at every point so far. It already carries the architecture of section 11. Then one
change at a time where the earlier sections point:

- `x-group`: GroupNorm instead of batch norm. Batch norm uses batch statistics in training but running averages in
  evaluation, and reconstruction and sampling both run in evaluation. GroupNorm is the same in both.
- `x-width64`: twice the channels. In section 11 the extra width fitted the training images better (train MSE
  0.00120 against 0.00151) but did not generalize; does it help the samples?

The better change (lower fitted-prior FID at seed 0) also gets seeds 1 and 2, so its row has error bars too.
About 30 minutes.

In [ ]:
# The rule, written before the X runs:
# 1. Build on the scored setting of sections 13 to 15 with the lowest mean FID from the fitted prior.
# 2. Try one change at a time where sections 11 and 12 point; seed 0 each. The change with the lower
#    fitted-prior FID gets seeds 1 and 2 as well.
SETTINGS = {name: {**B_BASE, **change} for name, change in B.items()}  # every scored setting so far
SETTINGS.update({f"final-beta{beta:g}": {**FINAL_BASE, "beta": beta} for beta in FINAL_BETAS})
SETTINGS.update({name: {**C_BASE, **change} for name, change in C.items()})
RUNS = {name: [name] for name in B}  # ... and its runs (several runs = seeds)
RUNS.update({f"final-beta{beta:g}": [f"final-beta{beta:g}-seed{seed}" for seed in FINAL_SEEDS] for beta in FINAL_BETAS})
RUNS.update({name: [f"{name}-seed{seed}" for seed in C_SEEDS] for name in C})
fitted_fid = {
    name: np.mean([score_run(run)["samples_aggregate"]["fid_2048"][0] for run in runs])
    for name, runs in RUNS.items()
}
X_FROM = min(fitted_fid, key=fitted_fid.get)
X_BASE = SETTINGS[X_FROM]
X = {  # name -> one change on X_BASE
    "x-group": {"norm": "group"},  # GroupNorm: the same normalization in training and evaluation
    "x-width64": {"hidden_channels": 64},  # twice the channels (section 11: a better fit that did not generalize)
}
for name, change in X.items():
    train_or_load(f"{name}-seed0", {**X_BASE, **change, "seed": 0})
    score_run(f"{name}-seed0")
X_BEST = min(X, key=lambda name: score_run(f"{name}-seed0")["samples_aggregate"]["fid_2048"][0])
for seed in (1, 2):
    train_or_load(f"{X_BEST}-seed{seed}", {**X_BASE, **X[X_BEST], "seed": seed})
    score_run(f"{X_BEST}-seed{seed}")
print(f"X builds on {X_FROM} (fitted-prior FID {fitted_fid[X_FROM]:.1f}); the better change: {X_BEST}")

rows = [seed_row(X_FROM, RUNS[X_FROM]) if len(RUNS[X_FROM]) > 1 else result_row(X_FROM)]
rows += [result_row(f"{name}-seed0") for name in X]
rows += [seed_row(X_BEST, [f"{X_BEST}-seed{seed}" for seed in (0, 1, 2)])]
print_table(rows, [*TABLE, "norm", "width", "is_fitted"])

# Part IV - Into the report

## 17. Every reported run in one table

`REPORT` lists the rows of the report table: a run name and the runs behind it (several runs = the seeds of one
setting: mean and std). Every run is scored from the prior and from the fitted prior, so all rows are measured the
same way. The table is printed, saved as CSV, and written as LaTeX to `report/generated/notebook/results.tex`
(read by `report/main.tex`).

In [ ]:
REPORT = {  # row of the report table -> its runs (several runs = the seeds of one setting: mean +- std)
    "vae-base": ["vae-base"],
    "vae-deep": ["vae-deep"],
    "vae-calibrated": ["vae-calibrated"],
    "a-stack": ["a-stack"],
    "r-aug": ["r-aug"],
    "b-beta1": ["b-beta1"],
    "b-calibrated": ["b-calibrated"],
    "final-beta1": [f"final-beta1-seed{seed}" for seed in FINAL_SEEDS],
    **{name: [f"{name}-seed{seed}" for seed in C_SEEDS] for name in C},
}
REPORT.setdefault(X_FROM, RUNS[X_FROM])  # the setting that section 16 builds on
for name in X:
    if name != X_BEST:
        REPORT[name] = [f"{name}-seed0"]  # retain the other normalization/width comparison
REPORT[X_BEST] = [f"{X_BEST}-seed{seed}" for seed in (0, 1, 2)]
report_table = {}
for label, names in REPORT.items():
    for name in names:
        score_run(name)  # every row is scored the same way (computed once, then cached)
    report_table[label] = seed_row(label, names) if len(names) > 1 else {**result_row(names[0]), "run": label}
columns = ["run", "K", "beta", "sigma", "aug", "drop", "val_mse", "kl", "fid_prior", "fid_fitted", "is_prior", "is_fitted"]
print_table(list(report_table.values()) + list(PCA_ROWS.values()), columns)
print(f"real images: FID-2048 {real_scores['fid_2048']:.1f}, IS {real_scores['inception_score']:.2f}")


def tex_number(row, column, digits, scale=1.0, spread=True):
    # the value; "mean $\pm$ std" for a row over seeds (spread=True); "--" when the row has no such number
    value = row.get(column, float("nan"))
    if value != value:  # NaN
        return "--"
    text = f"{value * scale:.{digits}f}"
    if spread and f"{column}_std" in row:
        text += f"$\\pm${row[f'{column}_std'] * scale:.{digits}f}"
    return text


lines = [
    r"\begin{tabular}{@{}lrrrrrrrr@{}}",
    r"\toprule",
    r" & & & & & \multicolumn{2}{c}{FID$_{2048}\downarrow$} & \multicolumn{2}{c}{IS$\uparrow$} \\",
    r"\cmidrule(lr){6-7}\cmidrule(lr){8-9}",
    r"Run & $K$ & $\beta$ & MSE$\times10^3$ & KL & prior & fitted & prior & fitted \\",
    r"\midrule",
]
for label, row in report_table.items():
    beta = "cal." if row["sigma"] == "calib" else f"{row['beta']:g}"
    numbers = [tex_number(row, "val_mse", 3, 1e3), tex_number(row, "kl", 0, spread=False)]
    numbers += [tex_number(row, c, 1) for c in ("fid_prior", "fid_fitted")]
    numbers += [tex_number(row, c, 2, spread=False) for c in ("is_prior", "is_fitted")]
    lines.append(f"\\texttt{{{label}}} & {row['K']} & {beta} & " + " & ".join(numbers) + r" \\")
lines.append(r"\midrule")
for row in PCA_ROWS.values():
    lines.append(
        f"PCA & {row['K']} & & {tex_number(row, 'val_mse', 3, 1e3)} & & {tex_number(row, 'fid_prior', 1)} & & "
        f"{tex_number(row, 'is_prior', 2)} & \\\\"
    )
lines.append(
    f"Real training images & & & & & {real_scores['fid_2048']:.1f} & & {real_scores['inception_score']:.2f} & \\\\"
)
lines += [r"\bottomrule", r"\end{tabular}"]
(REPORT_DIR / "results.tex").write_text("\n".join(lines) + "\n", encoding="utf-8")
with open(REPORT_DIR / "results.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(dict.fromkeys(k for row in report_table.values() for k in row)))
    writer.writeheader()
    writer.writerows(report_table.values())
print(f"wrote {REPORT_DIR / 'results.tex'} and results.csv")

## 18. Figures for the report

The reported model is the row with the lowest fitted-prior FID; the figures show its seed-0 run next to the
hand-written VAE (`Base`) and PCA. Left: reconstructions of four evenly spaced validation images. Right: four test
images and all four draws of every sampler with a fixed seed, nothing picked by hand. Below: the learning curves of
the reported run. Both go to `report/generated/notebook/`; then build the report:
`uv run python scripts/build_latex.py homeworks/hw1/report`.

In [ ]:
# The reported model: the report row with the lowest fitted-prior FID (mean over its seeds); its seed-0 run.
REPORTED = min(report_table, key=lambda label: report_table[label]["fid_fitted"])
best_model, best_run = load_vae(REPORT[REPORTED][0])
base_model, _ = load_vae("vae-base")
x4 = torch.stack([val[i][0] for i in (0, 158, 317, 475)])  # four validation images, evenly spaced
left = [  # reconstructions through the posterior mean
    ("Validation", x4),
    ("PCA-128", pca_reconstruct(x4.flatten(1).to(device), 128).view(-1, 1, SIZE, SIZE)),
    ("Base", base_model.reconstruct(x4.to(device))),
    ("Best", best_model.reconstruct(x4.to(device))),
]
with torch.random.fork_rng(devices=[device] if device.type == "cuda" else []):
    torch.manual_seed(0)
    right = [  # new images: all four draws, nothing picked by hand
        ("Test", test_images[torch.randperm(len(test_images))[:4]]),
        ("Base N(0,I)", base_model.sample(4)),
        ("Best N(0,I)", best_model.sample(4)),
        ("Best fitted", fitted_prior(best_model)(4)),
    ]
fig, axes = plt.subplots(4, 9, figsize=(7.2, 3.0), gridspec_kw={"width_ratios": [1, 1, 1, 1, 0.9, 1, 1, 1, 1]})
for r in range(4):
    axes[r, 4].axis("off")
    for block, start in ((left, 0), (right, 5)):
        label, images = block[r]
        for c in range(4):
            axes[r, start + c].imshow(images[c, 0].detach().cpu(), cmap="gray", vmin=0, vmax=1)
            axes[r, start + c].set(xticks=[], yticks=[])
        axes[r, start].set_ylabel(label, rotation=0, ha="right", va="center", fontsize=7)
axes[0, 0].set_title("Reconstructions", loc="left", fontsize=8)
axes[0, 5].set_title("New images", loc="left", fontsize=8)
fig.tight_layout(pad=0.3, w_pad=0.1, h_pad=0.2)
fig.savefig(REPORT_DIR / "images.png", dpi=300)
plt.show()

# Learning curves of the reported run; the dotted line is the best epoch (the kept weights)
history = best_run["history"]
epochs = range(1, len(history["train"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(7.2, 1.7))
for ax, term, title in zip(axes, ["neg_elbo", "kl"], ["negative ELBO", "KL"], strict=True):
    for split in ("train", "val"):
        ax.plot(epochs, [epoch[term] for epoch in history[split]], linewidth=1, label=split)
    later = [epoch[term] for split in ("train", "val") for epoch in history[split][len(epochs) // 10 :]]
    margin = 0.1 * (max(later) - min(later)) or 1.0
    ax.set_ylim(min(later) - margin, max(later) + margin)  # the first epochs are far off this scale
    ax.axvline(best_run["best_epoch"], color="gray", linestyle=":", linewidth=1)
    ax.set(title=f"{title} (nats per image)", xlabel="epoch")
axes[0].legend(fontsize=7)
fig.tight_layout(pad=0.3)
fig.savefig(REPORT_DIR / "curves.png", dpi=300)
plt.show()
(REPORT_DIR / "reported.txt").write_text(
    f"reported row: {REPORTED}\nruns: {REPORT[REPORTED]}\nfigures: seed-0 run {best_run['name']}\n"
    f"settings: {json.dumps(best_run['hparams'])}\n",
    encoding="utf-8",
)
macro = r"\newcommand{\ReportedRun}{\texttt{" + REPORTED + "}}\n"  # the caption in main.tex names the run
(REPORT_DIR / "reported.tex").write_text(macro, encoding="utf-8")
print(f"reported: {REPORTED} -> {best_run['name']}, best epoch {best_run['best_epoch']}")